<a href="https://colab.research.google.com/github/WMFong0/Python-Weather-Report-System/blob/Main/Python_Weather_App.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Hourly_Rainfall.py
import requests
import json

def get_hourlyRainfalldata():
  try:
    response = requests.get('https://data.weather.gov.hk/weatherAPI/opendata/hourlyRainfall.php?lang=en', timeout = 10)
    response.raise_for_status() # For HTTP error

    if not response:
      raise Exception("Received empty data from API")

    rainfalldata = response.json()
    return rainfalldata

  except Exception as e:
    raise Exception(f"Error getting hourlyRainfall data: {str(e)}")

def write_hourlyRainfall(data):
  result = {
        'value': data['value'] + " " + data['unit'],
        'station': data['automaticWeatherStation']
    }
  return result

def filter_hourlyRainfalldata():
  try:
    rainfalldata = get_hourlyRainfalldata()
    if not rainfalldata['hourlyRainfall']:
      raise Exception(f"Possible Hong Kong Observatory Full Maintenance. Try again later.")
      return

    result = {
      'data' : None,
      'Last Update': rainfalldata['obsTime'][11:19]
    }

    for data in rainfalldata['hourlyRainfall']:
      if (data['automaticWeatherStationID'] == 'RF001' and data['value'] != 'M'): #Second Priority location
        result['data'] = write_hourlyRainfall(data)
      elif (data['automaticWeatherStationID'] == 'RF019' and data['value'] != 'M'): #First Priority location
        if (data['value'] != 'M'):
          result['data'] = write_hourlyRainfall(data)
        break; #ter the function as data after is useless in this program

      if (not result['data']): # if second priority location is none, use these data first. First Priority location will replace them if available
        if (data['automaticWeatherStationID'] == 'RF002' and data['value'] != 'M'): result['data'] = write_hourlyRainfall(data)
        elif (data['automaticWeatherStationID'] == 'N12' and data['value'] != 'M'): result['data'] = write_hourlyRainfall(data)
    return result if (result['data']) else None
  except Exception as e:
    raise Exception(f"Error processing rainfall data. {str(e)}")


def print_hourlyRainfalldata(hourlyRainFalldata):
  if not hourlyRainFalldata:
    print("As all 4 nearest Automatic Weather Station is under maintance, we are sorry to inform you that last hour rainfall is not available.")
    return

  print(f"Latest Weather Update Time: {locale_data['hourlyRainFalldata']['Last Update']}")

  if not (hourlyRainFalldata['data']['station'] in ["Tuen Mun", "Lau Fau Shan"]):
    print("Due to maintance, we are unable to present you last hour rainfall from Tuen Mun. \nWe will give you the result from other nearest automatic weather station")

  print(f"In {hourlyRainFalldata['data']['station']}, the RainFall in last hour is {hourlyRainFalldata['data']['value']}")



In [ ]:
# Main.py
import requests
import json
import datetime


locale_data = {
    'hourlyRainFalldata': None
}

current_datetime = datetime.datetime.now().astimezone(datetime.timezone(datetime.timedelta(hours=8))); # Enforce Hong Kong Timezone
try:
  locale_data['hourlyRainFalldata'] = filter_hourlyRainfalldata()
except Exception as e:
  print(f"{str(e)}\n\n")

print("Welcome using Weather Report System." + "\n" +
      "This system uses Hong Kong Observatory Data to report")

print("\n"*2 + "="*(20+len("Weather Report")+20) + "\n" + "="*20 + "Weather Report" + "="*20 + "\n")

print(current_datetime.strftime('Today is %Y/%m/%d. \nCurrent Time: %H:%M:%S'))

print_hourlyRainfalldata(locale_data['hourlyRainFalldata'])

Welcome using Weather Report System.
This system uses Hong Kong Observatory Data to report


====================Weather Report====================

Today is 2025/07/05. 
Current Time: 17:06:28
Latest Weather Update Time: 16:45:00
In Tuen Mun, the RainFall in last hour is 0 mm
